In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/home/dataopske/Desktop/jav/data/raw/costofliving/nairobi_cost_of_living.csv')

In [3]:
df.head()

,Date,Area,Rent,Food,Transport,Utilities,Misc,Total
0,2019-01-31,Westlands,90341,25454,15634,6142,13468,151039
1,2019-01-31,Westlands,93902,24808,17194,6803,13608,156315
2,2019-01-31,Kileleshwa,78160,23302,13104,6454,12959,133979
3,2019-01-31,Westlands,87721,29735,18420,7934,14908,158719
4,2019-01-31,Ngong,32881,13471,10989,3062,8450,68853


In [10]:
df['Area'].nunique()

19

In [5]:
# ensure the column is parsed as datetime
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# assume df has ['Date', 'Area', 'Total']
latest = df[df['Date'].dt.year == 2023]
area_income = latest.groupby('Area')['Total'].mean().reset_index()

- We use monthly total expenditure as a proxy for disposable income requirements. Neighborhoods with higher living costs imply residents with greater income capacity, given similar consumption baskets.

In [6]:
# classify into tiers (quantiles)
area_income['Income_Class'] = pd.qcut(
    area_income['Total'],
    q=3,
    labels=['Low', 'Middle', 'High']
)

In [7]:
area_income

,Area,Total,Income_Class
0,Donholm,76838.006568,Middle
1,Embakasi,59559.292929,Middle
2,Githurai,34393.532526,Low
3,Juja,41348.835749,Low
4,Karen,238099.479806,High
5,Kasarani,50899.111952,Low
6,Kikuyu,50964.187602,Low
7,Kileleshwa,142338.459504,High
8,Kitengela,59942.812698,Middle
9,Langata,110996.670866,High


In [8]:
area_income.shape

(19, 3)

In [11]:
area_income['Income_Class'].value_counts()

Income_Class
Low       7
Middle    6
High      6
Name: count, dtype: int64

In [13]:
area_income.to_csv('/home/dataopske/Desktop/jav/data/processed/nairobi_area_income.csv', index=False)

In [42]:
# wards_nbo = pd.read_csv('/home/dataopske/Desktop/jav/data/processed/wards_nbo.csv')
df1 = pd.read_csv('/home/dataopske/Desktop/jav/data/processed/wards_nbo_pop.csv')

In [44]:
df1.head(1)

,gid,pop2009,county,subcounty,ward,uid,scuid,cuid,geometry,population,area_km2,pop_density
0,2044,55158.0,Nairobi,Ruaraka Sub County,Mathare North Ward,dOQTHZsuaIb,Cc8uEFkzfVf,jkG3zaihdSs,"POLYGON ((36.876525595436 -1.2520951959819777,...",74177.992829,0.463546,160022.773428


In [40]:
df2 = pd.read_csv('/home/dataopske/Desktop/jav/data/raw/costofliving/cost_of_living_scraped.csv')

In [45]:
df2.head(1)

,Sub-County,Ward,"Est. Median Household Expenditure (KSh/month, 2025 adj.)",Poverty Rate (%),Income Class,Notes/Key Proxy Source
0,Dagoretti North,Kangemi,20017,69,Low,KNBS 2024 / KIHBS 2021 (2025 adj.)


In [49]:
df1 = df1.rename(columns={'ward': 'Ward'})

In [52]:
import pandas as pd
import numpy as np
from difflib import get_close_matches

# Assuming df1 and df2 are already loaded
# (df1 from your CSV with columns like Sub-County, Ward, etc.)
# df2 with 'Ward' column (and possibly others)

def clean_ward(name):
    cleaned = str(name).lower().strip()
    cleaned = cleaned.replace(' ward', '').replace('ward', '')
    cleaned = cleaned.replace('/', ' ').replace("'", '').replace('  ', ' ')
    return cleaned

df1['Ward_clean'] = df1['Ward'].apply(clean_ward)
df2['Ward_clean'] = df2['Ward'].apply(clean_ward)

# Exact merge
exact_merge = df1.merge(df2, on='Ward_clean', how='left', indicator=True, suffixes=('_df1', '_df2'))
exact_merge['df2_ward'] = exact_merge['Ward_df2']
exact_merge['match_type'] = np.where(exact_merge['_merge'] == 'both', 'exact', pd.NA)
exact_merge = exact_merge.rename(columns={'Ward_df1': 'Ward'})
exact_merge = exact_merge.drop(columns=['Ward_clean', '_merge', 'Ward_df2'], errors='ignore')

df_joined = exact_merge

# Fuzzy matching for unmatched rows
unmatched_mask = df_joined['df2_ward'].isna()
unmatched_cleans = df_joined.loc[unmatched_mask, 'Ward'].apply(clean_ward).unique()

fuzzy_mapping = {}
for w1_clean in unmatched_cleans:
    matches = get_close_matches(w1_clean, df2['Ward_clean'].unique(), n=1, cutoff=0.8)
    if matches:
        fuzzy_clean = matches[0]
        fuzzy_original = df2[df2['Ward_clean'] == fuzzy_clean]['Ward'].iloc[0]
        fuzzy_mapping[w1_clean] = fuzzy_original

# Apply fuzzy
df_joined.loc[unmatched_mask, 'Ward_fuzzy_clean'] = df_joined.loc[unmatched_mask, 'Ward'].apply(clean_ward)
fuzzy_applied = df_joined['Ward_fuzzy_clean'].map(fuzzy_mapping)
df_joined['df2_ward'] = df_joined['df2_ward'].fillna(fuzzy_applied)

# Create fill series for match_type (fixes ndarray issue)
fill_series = pd.Series(np.where(fuzzy_applied.notna(), 'fuzzy', 'unmatched'), index=df_joined.index)
df_joined['match_type'] = df_joined['match_type'].fillna(fill_series)

# Clean up temp columns
df_joined = df_joined.drop(columns=['Ward_fuzzy_clean'], errors='ignore')

# If df2 has other columns you want to keep (e.g., data from df2), they are already merged with _df2 suffix.
# Drop them if not needed: df_joined = df_joined.drop(columns=[col for col in df_joined.columns if col.endswith('_df2')], errors='ignore')

# Display or save
print(df_joined.head())
df_joined.to_csv('joined_wards_fixed.csv', index=False)

    gid  pop2009   county                   subcounty  \
0  2044  55158.0  Nairobi         Ruaraka  Sub County   
1  2047  70641.0  Nairobi   Embakasi South Sub County   
2  2008  38384.0  Nairobi       Westlands  Sub County   
3  2012  43122.0  Nairobi  Dagoretti North Sub County   
4  2015  27202.0  Nairobi  Dagoretti North Sub County   

                       Ward          uid        scuid         cuid  \
0        Mathare North Ward  dOQTHZsuaIb  Cc8uEFkzfVf  jkG3zaihdSs   
1          Imara Daima Ward  Nt7PPe0Vdou  aDp1odOWYC1  jkG3zaihdSs   
2  Parklands/highridge Ward  QhDd2LAuXAF  f1T0Ltob8VQ  jkG3zaihdSs   
3             Kilimani Ward  yXbOIljEz90  CcTr4bcVGAG  jkG3zaihdSs   
4           Kileleshwa Ward  DnVTXbCnup5  CcTr4bcVGAG  jkG3zaihdSs   

                                            geometry     population  \
0  POLYGON ((36.876525595436 -1.2520951959819777,...   74177.992829   
1  POLYGON ((36.87933769794595 -1.310366334321338...   92557.211478   
2  POLYGON ((36.8071198

In [53]:
df_joined

,gid,pop2009,county,subcounty,Ward,uid,scuid,cuid,geometry,population,area_km2,pop_density,Sub-County,"Est. Median Household Expenditure (KSh/month, 2025 adj.)",Poverty Rate (%),Income Class,Notes/Key Proxy Source,df2_ward,match_type
0,2044,55158.0,Nairobi,Ruaraka Sub County,Mathare North Ward,dOQTHZsuaIb,Cc8uEFkzfVf,jkG3zaihdSs,"POLYGON ((36.876525595436 -1.2520951959819777,...",74177.992829,0.463546,160022.773428,NaN,NaN,NaN,NaN,NaN,NaN,unmatched
1,2047,70641.0,Nairobi,Embakasi South Sub County,Imara Daima Ward,Nt7PPe0Vdou,aDp1odOWYC1,jkG3zaihdSs,POLYGON ((36.87933769794595 -1.310366334321338...,92557.211478,3.914392,23645.362517,Embakasi South,47581.0,15.0,Middle,KNBS 2024 / KIHBS 2021 (2025 adj.),Imara Daima,exact
2,2008,38384.0,Nairobi,Westlands Sub County,Parklands/highridge Ward,QhDd2LAuXAF,f1T0Ltob8VQ,jkG3zaihdSs,POLYGON ((36.80711989990031 -1.249858130447110...,69612.843253,8.180886,8509.205750,NaN,NaN,NaN,NaN,NaN,NaN,unmatched
3,2012,43122.0,Nairobi,Dagoretti North Sub County,Kilimani Ward,yXbOIljEz90,CcTr4bcVGAG,jkG3zaihdSs,POLYGON ((36.78250188847331 -1.266234196060095...,142658.597296,16.060785,8882.417364,Westlands,52029.0,16.0,Middle,KNBS 2024 / KIHBS 2021 (2025 adj.),Kilimani,exact
4,2015,27202.0,Nairobi,Dagoretti North Sub County,Kileleshwa Ward,DnVTXbCnup5,CcTr4bcVGAG,jkG3zaihdSs,POLYGON ((36.78027179110818 -1.258881522540643...,81906.408407,9.039389,9061.056220,NaN,NaN,NaN,NaN,NaN,NaN,unmatched
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,2087,28260.0,Nairobi,Mathare Sub County,Mabatini Ward,rD8Z7AgYLaS,gh2kzpOFCeF,jkG3zaihdSs,"POLYGON ((36.86431834562677 -1.25890658573048,...",48278.476367,0.354188,136307.362110,NaN,NaN,NaN,NaN,NaN,NaN,unmatched
83,2089,34523.0,Nairobi,Mathare Sub County,Ngei Ward,tsmsPb6qMFa,gh2kzpOFCeF,jkG3zaihdSs,POLYGON ((36.87267267353588 -1.254062049868481...,0.000000,0.434336,0.000000,Mathare,45192.0,25.0,Middle,KNBS 2024 / KIHBS 2021 (2025 adj.),Ngei,exact
84,2090,38374.0,Nairobi,Mathare Sub County,Mlango Kubwa,lvETWuhiIMi,gh2kzpOFCeF,jkG3zaihdSs,POLYGON ((36.84714599251155 -1.263204203703432...,0.000000,0.412475,0.000000,Mathare,91792.0,6.0,High,KNBS 2024 / KIHBS 2021 (2025 adj.),Mlango Kubwa,exact
85,2091,33824.0,Nairobi,Mathare Sub County,Kiamaiko Ward,tgAB60BL0u9,gh2kzpOFCeF,jkG3zaihdSs,POLYGON ((36.87860863322339 -1.251458252751912...,74514.220482,0.668185,111517.368470,Mathare,34282.0,17.0,Middle,KNBS 2024 / KIHBS 2021 (2025 adj.),Kiamaiko,exact
